# AsymmeTree: A Flexible Python Library for Simulating Gene Family Histories

## 1. Introduction – What are Gene Family Histories and why simulate them?

### Evolutionary events:

- **Duplication:** a gene is copied; the copies (paralogs) can take on new functions.
- **Loss:** a gene disappears from a lineage.
- **Horizontal gene transfer (HGT):** a gene moves from one species to another non‑ancestral species (common in bacteria).
- **Gene conversion:** a gene is replaced by a homologous copy within the same species.

## 2. Related Work (comparison with other tools)

- **Species tree simulators:** TreeSim, TreeSimGM, TESS, castor (in R). AsymmeTree implements similar methods plus the innovation model (unique).
- **Sequence simulators:** Seq‑Gen (fast but no indels), INDELible (with indels, in C++), Pyvolve (Python, no indels). AsymmeTree includes its own sequence simulator with indels and is faster than Pyvolve.
- **Full gene family simulators:** GenPhyloData, SaGePhy, SimPhy, Zombi. Some do not generate sequences or call external programs. AsymmeTree integrates everything (species, genes, sequences) and explicitly models neofunctionalisation/subfunctionalisation.

## 3. Species tree simulation

### Available models:

- **Yule model (pure birth):** only speciations, no extinctions.
- **Constant‑rate birth‑death process (BDP):** with constant speciation and extinction rates.
- **Episodic BDP (EBDP):** rates can change in temporal "episodes"; mass extinctions can occur.

### Sampling conditions:

The tree can be generated conditioned on:

- A fixed number of extant species.
- The total tree age (time from root to leaves).
- Both.

### Innovation model

(alternative to random lineage choice for speciation): each species has a set of "features"; speciation occurs when a feature is gained or lost. This can produce more realistic trees according to some studies.

### Polytomies (multifurcations)

Normally trees are bifurcating (each internal node has two children). Polytomies can be introduced either randomly (with probability $p$) or with a bias (younger edges are more likely to be collapsed).

### Code Example – Species Tree Simulation

The function `species_tree_n()` simulates a species tree with a fixed number of extant species:

In [ ]:
import asymmetree.treeevolve as te
from asymmetree.utils.phylogenetic_trees import to_newick

# Yule model (pure birth) with 10 extant species
tree = te.species_tree_n(10, model="yule", birth_rate=1.0)
print(to_newick(tree))

# Episodic BDP with two episodes: (time, birth_rate, death_rate, sampling_fraction)
tree2 = te.species_tree_n(10, model="EBDP", episodes=[(1.0, 0.3, 0.8, 0.0), (0.9, 0.4, 0.6, 0.3)])
print(to_newick(tree2))

# Simulate conditioned on tree age (2.0 time units)
tree3 = te.species_tree_age(2.0, model="yule", birth_rate=1.0)
print(to_newick(tree3))

# Simulate conditioned on both number of species AND age
tree5 = te.species_tree_n_age(10, 1.0, model="yule", birth_rate=1.0)
print(to_newick(tree5))

((((((14:0.3913919868235878,15:0.3913919868235878)13:0.12380914038260737,12:0.5152011272061952)7:0.8312988711833307,(18:0.2629101593157652,19:0.2629101593157652)6:1.0835898390737606)5:0.12761384411438348,((10:0.5475909750819823,11:0.5475909750819823)8:0.7939247844293646,(16:0.3020443940585862,17:0.3020443940585862)9:1.0394713654527608)4:0.13259808299256237)2:1.3629769937735317,3:2.837090836277441)1:0.6412425789320233)0:0.0;
(((((((((((0:0.60471332630397,3:0.60471332630397)29:0.2675074103305739,(18:0.20797417895895337,18:0.20797417895895337)26:0.36424655767559055)33:0.06172880121363522,16:0.7612842145175018)34:0.1347662970261122,(10:0.5229121044572027,18:0.22291210445720272)28:0.5458037304170886)37:0.8083854397823196,(((5:0.6289179243244348,((6:0.1659318599776063,9:0.1659318599776063)15:0.05637349937227412,12:0.21399616414052225)17:0.40661256497455445)30:0.32079017331481086,18:0.6497080976392458)35:0.5278209008262958,39:0.05959787011665152)40:0.3995722761910694)42:0.02634267892262976,((

## 4. Gene tree simulation

**Process**: starting with a single gene at the root of the species tree, we move forward in time simulating events according to **constant rates** (user‑defined). Waiting times are drawn from exponential distributions.

**Event types**:
- **Duplication**: one gene gives rise to $2 + k$ copies. By default $k=0$ (bifurcation). If $k>0$, a polytomy is introduced in the gene tree (e.g., due to concerted evolution). The value of $k$ is drawn from a Poisson distribution with parameter $\lambda$ (user‑defined; default $\lambda=0$).
- **Loss**: a copy is removed.
- **HGT (horizontal gene transfer)**:
  - **Additive**: a copy of the gene is added to another species (the original is kept). The recipient is chosen among coexisting species.
  - **Replacing**: the transferred copy replaces a homologous gene in the recipient species, and the original is lost.
- **Gene conversion**: similar to replacing HGT but within the same species.
- **Speciations and extinctions**: speciations duplicate all genes; extinctions remove all genes in that lineage.

**Distance bias in HGT**: transfers can be made more likely between phylogenetically close species/genes. Options:
- No bias (uniform)
- Inverse: $1/(\alpha t)$
- Exponential: $e^{-\alpha t}$
where $t$ is time since the last common ancestor.

**Essential genes**: option to prevent complete extinction of a gene family within a species (the last copy cannot be lost). Models genes whose absence is lethal.

### Code Example – Gene Tree Simulation

The function `dated_gene_tree()` simulates a gene tree along an existing species tree:

In [2]:
import asymmetree.treeevolve as te
import asymmetree.seqevolve as se
from asymmetree.utils.phylogenetic_trees import to_newick
from asymmetree.genome import GenomeSimulator
import matplotlib.pyplot as plt

In [ ]:
# First, generate a species tree
S = te.species_tree_n_age(10, 1.0)

# Then simulate a gene tree along it
TGT = te.dated_gene_tree(
    S,
    dupl_rate=1.0,      # duplication rate
    loss_rate=1.0,      # loss rate
    hgt_rate=0.2,       # horizontal gene transfer rate
    gc_rate=0.2,        # gene conversion rate
    prohibit_extinction="per_species"  # prevent complete loss in each species
)
print(to_newick(TGT))

(((5<1-2>:0.010040185646168709,(((((19<6-8>:0.0580914334195628,((((78<18>:0.03045059181013276,79<19>:0.03045059181013276)68:0.09211846786409003,(80<18>:0.03045059181013276,81<19>:0.03045059181013276)69:0.09211846786409003)42:0.09104347989499742,(76<18>:0.03045059181013276,77<19>:0.03045059181013276)43:0.18316194775908745)36:0.0007358419962266993,37<10>:0.21434838156544692)18:0.3947607194601698)16:0.011479946492228277,(21<6-8>:0.21136226652102308,(39<9-10>:0.035337277270546835,(74<18>:0.03045059181013276,75<19>:0.03045059181013276)38:0.18389778975531415)20:0.3947607194601698)17:0.011479946492228277)10:0.09406033261442293,(((((((72<14>:0.04887905687991298,73<14>:0.04887905687991298)52:0.09959588021229035,53<15>:0.14847493709220333)40:0.0658679809423511,(54<8-14>:0.07612541970727754,55<15>:0.14847493709220333)41:0.0658679809423511)30:0.1870079873171919,((70<15>:0.09677524810752103,71<15>:0.09677524810752103)51:0.0516996889846823,50<8-14>:0.09959588021229035)31:0.252875968259543)29:0.09449

## 5. Evolutionary rate heterogeneity

**Problem**: After step 2, distances in the gene tree are just times (strict molecular clock). In reality, different lineages evolve at different speeds.

**Three levels of rate multipliers**:

1. **Gene‑family‑specific baseline**: all branches of the same gene tree share a baseline factor drawn from a distribution (uniform, Gamma, exponential).

2. **Species‑specific (lognormal relaxed clock)**:
   - Rates are assigned to each node of the species tree.
   - The root has rate $1$.
   - For each edge (parent $u \to$ child $v$), $\log(r_v)$ is drawn from a normal distribution with variance $\beta t$ (where $t$ is divergence time) and mean adjusted so that the expected value of $r_v$ equals $r_u$ (no bias):
     $$
     \log(r_v) \sim \mathcal{N}\left(\text{mean adjusted so that } \mathbb{E}[r_v] = r_u,\; \beta t\right)
     $$
   - The rate for the edge is then computed as:
     $$
     r_{\text{edge}} = \frac{r_u + r_v}{2}
     $$
   - Parameter $\beta$ controls how much rates vary: if $\beta=0$, all rates are $1$ (strict clock).

3. **Paralog‑specific (asymmetry after duplication)**:
   - Models **neofunctionalisation** and **subfunctionalisation**.
   - **Neofunctionalisation**: after duplication, one copy acquires a new function and evolves faster (divergent), the other remains conserved.
   - **Subfunctionalisation**: both copies specialise in subfunctions and both evolve faster.
   - User defines weights for each mode.
   - "Divergent" branches receive a multiplier $1 + x$, where $x$ follows a Gamma distribution fitted to real yeast data ($k=0.5$, $\theta=2.2$). This produces the asymmetry parameter:
     $$
     R' = \frac{\max(K_a, K_b)}{\min(K_a, K_b)} = 1 + X
     $$
   - If a divergent branch becomes the only surviving copy in its species (due to loss of the other), its rate is reset to $1$ (conserved).

**Final effect**: evolutionary distance on a branch = time $\times$ (product of applicable multipliers, summing if multiple periods exist).

### Code Example – Rate Heterogeneity

The function `rate_heterogeneity()` introduces the three levels of rate variation:

In [1]:
TGT = te.rate_heterogeneity(
    TGT,                           # the gene tree
    S,                             # the species tree
    base_rate=1.0,                 # gene-family-specific baseline
    autocorr_variance=0.2,         # β parameter for lognormal relaxed clock
    rate_increase=("gamma", 0.5, 2.2),  # Gamma distribution for paralog asymmetry
    CSN_weights=(1, 1, 1),         # weights for (C)onserved, (S)ubfunctionalisation, (N)eofunctionalisation
)
print(to_newick(TGT))

NameError: name 'te' is not defined

## 6. Pruning of loss branches

**Process**: branches that lead only to losses are removed because they are useless for sequence simulation. The planted root is also suppressed. The result is a **pruned gene tree** with all leaves being extant genes in current species.

### Code Example – Pruning

In [ ]:
PGT = te.prune_losses(TGT)
print("------------- PGT -------------")
print(to_newick(PGT))

------------- PGT -------------
(((((((((78<18>:0.0282371888321431,79<19>:0.028498175235056166)68:0.1479557959628206,(80<18>:0.03358217122591247,81<19>:0.03521711990324371)69:0.0611038519696244)42:0.17017701629693816,(76<18>:0.054524128744730845,77<19>:0.022408356347476635)43:0.35395400280836725)36:0.0004316885606303484,37<10>:0.124051971810542)18:0.2940851708646947,(74<18>:0.01753583813258656,75<19>:0.018960956516970773)38:0.4019705220265717)10:0.07755842055625482,(((((72<14>:0.03541865772285455,73<14>:0.03464911236709464)52:0.08005966839711998,53<15>:0.1438554405646144)40:0.09998865252750616,55<15>:0.22357197157781775)30:0.1586124735579838,(70<15>:0.9787543217568634,71<15>:0.20152953418119118)51:0.3448240344916263)29:0.11976925950968191,48<14>:0.567915235739697)25:0.17415253283654764)8:0.010037614988708098,(60<7>:0.09135353415369891,61<7>:0.295124680689204)58:0.7727813059250146)6:0.007232873754623911,(((82<12>:0.06131441522731494,83<12>:0.23361469306052998)66:0.15024913098496256,67<1

## 7. Sequence simulation

**Input**: pruned gene tree with evolutionary distances (expected number of substitutions per site).

**Substitution models**:
- **Nucleotide**: JC69, K80, GTR.
- **Amino acid**: Dayhoff, BLOSUM62, JTT, WAG, LG, or custom models in PAML format.
- The substitution probability matrix for a branch of length $t$ is:
  $$
  P = e^{Q t}
  $$
  where $Q$ is the rate matrix derived from the substitution model.

**Indel model** (insertions and deletions):
- Continuous‑time Markov process. The duration of the process is the evolutionary distance of the branch.
- Indel lengths follow a Zipfian distribution (default parameter $1.821$, fitted to real data).

**Among‑site rate heterogeneity**:
- **$+\Gamma$ model**: each site (or group of sites) gets a rate multiplier drawn from a Gamma distribution with mean $1$ and shape $\alpha$ (smaller $\alpha$ = more heterogeneity).
- **Invariant sites ($+\bar{I}$)**: a proportion $p$ of sites never mutate.

**Algorithms for sequence evolution**:
- **Substitution probability matrix**: compute $P = e^{Q t}$ and apply it to the whole sequence. Fast, but becomes heavy when many site‑specific rates exist.
- **Gillespie algorithm**: simulates mutations site by site, slower but necessary when high among‑site rate heterogeneity is used.

**True alignment**: AsymmeTree assigns unique identifiers to each new site (from insertions) and inherits them, so it can reconstruct which columns of the multiple sequence alignment are homologous (the "true" alignment, even with indels).

### Code Example – Sequence Simulation

The `Evolver` class handles sequence evolution along a tree:

In [3]:
# Define substitution model (WAG for amino acids)
subst_model = se.SubstModel("a", "WAG")

# Define indel model: insertion rate 0.01, deletion rate 0.01, Zipfian length distribution
indel_model = se.IndelModel(0.01, 0.01, length_distr=("zipf", 1.821))

# Initialize the evolver
evolver = se.Evolver(subst_model, indel_model=indel_model, gillespie=False)

# Simulate a tree (5 species, age 1.0)
T = te.species_tree_n_age(5, 1.0)

# Evolve sequences along the tree, starting with length 150
evolver.evolve_along_tree(T, start_length=150)

# Write sequences to FASTA file (including inner nodes)
evolver.write_sequences("testfile.fasta", include_inner=True)

# Generate the true multiple sequence alignment
alg_seq = evolver.true_alignment(write_to="testfile.alignment")

## 8. Additional Feature – Genome Simulation

AsymmeTree can also simulate whole genomes/proteomes with multiple gene families.

### Code Example – Genome Simulation

In [ ]:
from asymmetree.treeevolve import species_tree_n_age
from asymmetree.genome import GenomeSimulator
from asymmetree.seqevolve import SubstModel, IndelModel

# Generate a species tree
species_tree = species_tree_n_age(10, 1.0, model="yule")

# Define models
subst_model = SubstModel("a", "JTT")
indel_model = IndelModel(0.01, 0.01, length_distr=("zipf", 1.821))

# Initialize genome simulator
genome_sim = GenomeSimulator(species_tree, outdir="testfile_genome")

# Simulate 50 gene families along the species tree
genome_sim.simulate_gene_trees(
    50,
    dupl_rate=1.0,
    loss_rate=0.5,
    base_rate=("gamma", 1.0, 1.0),
    prohibit_extinction="per_species",
)

# Simulate sequences for all gene families
genome_sim.simulate_sequences(
    subst_model,
    indel_model=indel_model,
    het_model=None,
    length_distr=("constant", 200)
)

## 9. Validation experiments (results)

The authors show that their simulation is correct through comparisons:

- **Species trees:** compared with TreeSim – no significant differences (Mann‑Whitney U test, $p > 0.05$).
- **HGT distance bias:** when using inverse or exponential bias, the times between donor and recipient become smaller, as expected.
- **Lognormal model:** they verify that leaf rate logarithms follow a normal distribution and that there is no bias towards higher or lower rates.
- **Paralog asymmetry ($R'$):** simulated values fit the original Gamma distribution from yeast data.
- **Phylogenetic distance re‑estimation:** sequences are simulated with JC69, K80, WAG, JTT; then distances are re‑estimated. The results correlate well with true distances (Figs. 8‑10).
- **Execution time:** AsymmeTree is slower than C/C++ tools (Seq‑Gen, INDELible) but faster than Pyvolve (another Python package).